In [27]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS


In [75]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")


In [78]:



clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

clip_model.eval()


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 30491.77it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05, eleme

In [52]:
# Embedding Functions

def embed_image(image_data):
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt")

    with torch.no_grad():
        vision_outputs = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        features = clip_model.visual_projection(vision_outputs.pooler_output)
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

def embed_text(text):
    inputs = clip_processor(text=text, return_tensors="pt", padding=True, truncation=True, max_length=77)

    with torch.no_grad():
        text_outputs = clip_model.text_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
        features = clip_model.text_projection(text_outputs.pooler_output)
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()


In [53]:
pdf_path = "notbooks/multimodal_sample.pdf"
doc = fitz.open(pdf_path)

all_docs = []
all_embeddings = []
image_data_source = {}
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)


In [49]:
doc

Document('notbooks/multimodal_sample.pdf')

In [54]:
all_docs = []
all_embeddings = []
image_data_source = {}

with fitz.open(pdf_path) as doc:
    for i, page in enumerate(doc):
        text = page.get_text()
        if text.strip():
            chunks = splitter.split_text(text)
            for chunk in chunks:
                all_docs.append(Document(page_content=chunk, metadata={"page": i, "type": "text"}))
                all_embeddings.append(embed_text(chunk))

        image_list = page.get_images(full=True)
        for img_index, img in enumerate(image_list):
            try:
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
                image_id = f"page_{i}_img_{img_index}"
                buffered = io.BytesIO()
                pil_image.save(buffered, format="PNG")
                img_base64 = base64.b64encode(buffered.getvalue()).decode()
                image_data_source[image_id] = img_base64
                image_embedding = embed_image(pil_image)
                # Append doc and embedding together to keep lists aligned
                all_docs.append(Document(page_content=f"[Image: {image_id}]",
                                         metadata={"page": i, "type": "image", "image_id": image_id}))
                all_embeddings.append(image_embedding)
            except Exception as e:
                print(f"Error processing image on page {i}: {e}")
                continue

print(f"Total docs: {len(all_docs)} | Images: {sum(1 for d in all_docs if d.metadata['type'] == 'image')}")


Total docs: 2 | Images: 1


In [55]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [56]:
text_docs = [(doc, emb) for doc, emb in zip(all_docs, all_embeddings) if doc.metadata["type"] == "text"]
image_docs = [(doc, emb) for doc, emb in zip(all_docs, all_embeddings) if doc.metadata["type"] == "image"]

text_embeddings_array = np.array([emb for _, emb in text_docs])   # shape: (N, 384)
image_embeddings_array = np.array([emb for _, emb in image_docs])  # shape: (M, 512)

print(f"Text docs: {len(text_docs)}, embedding shape: {text_embeddings_array.shape}")
print(f"Image docs: {len(image_docs)}, embedding shape: {image_embeddings_array.shape}")


Text docs: 1, embedding shape: (1, 512)
Image docs: 1, embedding shape: (1, 512)


In [58]:
embeddings_array = np.array(all_embeddings)
all_docs, embeddings_array

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
  Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')],
 array([[-0.00267245,  0.01282999, -0.05183139, ..., -0.00385089,
          0.02977717, -0.00010682],
        [ 0.01732341, -0.0132769 , -0.02427026, ...,  0.08994047,
         -0.00272155,  0.03253042]], shape=(2, 512), dtype=float32))

In [62]:
vector_store = FAISS.from_embeddings(text_embeddings= [(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
                    embedding = None,
                    metadatas=[doc.metadata for doc in all_docs])


vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [82]:
# Initialize GPT-4 Vision model
llm = init_chat_model("openai:gpt-4.1")
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000247E4CFEA20>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000247E6751460>, root_client=<openai.OpenAI object at 0x00000247E6DBF2C0>, root_async_client=<openai.AsyncOpenAI object at 0x00000247E297D8E0>, model_name='gpt-4.1', model_kwargs={}, openai_

In [67]:
def retreive_multimodal(query,top_k=3):
    query_embedding = embed_text(query)
    # Compute cosine similarity with text and image embeddings
    results = vector_store.similarity_search_by_vector(query_embedding, k=top_k)
    return results

In [71]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []
    
    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_source:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_source[image_id]}"
                }
            })
    
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [72]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retreive_multimodal(query, top_k=5)
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get response from GPT-4V
    response = llm.invoke([message])
    
    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [83]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************jowA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}